In [0]:
%run "/Workspace/Users/sufianaslam127@gmail.com/audit_helper"

In [0]:
from datetime import datetime, timezone

run_start = datetime.now(timezone.utc)

print("Inventory Silver audit run started.")

Inventory Silver audit run started.


In [0]:
from pyspark.sql.functions import (
    col,
    to_timestamp,
    lit,
    when,
    unix_timestamp
)

from pyspark.sql.types import IntegerType

[audit] logged run 858da511-6e80-45b0-8e19-6dffbc837814 (success), latency=0.0s


In [0]:
bronze_path = "abfss://bronze@adlsnexpulse01.dfs.core.windows.net/inventory/"

silver_path = "abfss://silver@adlsnexpulse01.dfs.core.windows.net/inventory/"

quarantine_path = "abfss://silver@adlsnexpulse01.dfs.core.windows.net/_quarantine/inventory/"

In [0]:
bronze_df = (
    spark.read
    .format("delta")
    .load(bronze_path)
)

bronze_count = bronze_df.count()

print(f"Bronze Inventory: {bronze_count}")

Bronze Inventory: 60


In [0]:
typed_df = (
    bronze_df
    .withColumn(
        "stock_quantity",
        col("stock_quantity").cast(IntegerType())
    )
    .withColumn(
        "event_timestamp",
        to_timestamp(col("timestamp"))
    )
)

In [0]:
validated_df = typed_df.withColumn(
    "failure_reason",
    when(
        col("event_id").isNull(),
        lit("missing_event_id")
    )
    .when(
        col("warehouse_id").isNull(),
        lit("missing_warehouse_id")
    )
    .when(
        col("stock_quantity").isNull() | (col("stock_quantity") < 0),
        lit("invalid_stock_quantity")
    )
    .when(
        col("event_timestamp").isNull(),
        lit("unparseable_timestamp")
    )
    .otherwise(lit(None))
)

In [0]:
validated_df = validated_df.withColumn(
    "ingestion_delay_seconds",
    unix_timestamp(col("bronze_loaded_at"))
    - unix_timestamp(col("event_timestamp"))
)

LATE_THRESHOLD_SECONDS = 120

validated_df = validated_df.withColumn(
    "is_late",
    col("ingestion_delay_seconds") > LATE_THRESHOLD_SECONDS
)

In [0]:
pre_dedup_valid_df = validated_df.filter(
    col("failure_reason").isNull()
)

quarantine_df = validated_df.filter(
    col("failure_reason").isNotNull()
)

pre_dedup_valid_count = pre_dedup_valid_df.count()
quarantined_count = quarantine_df.count()

print(f"Valid before dedup: {pre_dedup_valid_count}")
print(f"Quarantined:        {quarantined_count}")

Valid before dedup: 59
Quarantined:        1


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

inventory_window = (
    Window
    .partitionBy("product_id", "warehouse_id")
    .orderBy(
        col("event_timestamp").desc(),
        col("bronze_loaded_at").desc(),
        col("event_id").desc()
    )
)

final_valid_df = (
    pre_dedup_valid_df
    .withColumn("row_num", row_number().over(inventory_window))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

final_valid_count = final_valid_df.count()

duplicates_removed_count = (
    pre_dedup_valid_count - final_valid_count
)

print(f"Pre-dedup valid:   {pre_dedup_valid_count}")
print(f"Final inventory:   {final_valid_count}")
print(f"State rows removed: {duplicates_removed_count}")

Pre-dedup valid:   59
Final inventory:   55
State rows removed: 4


In [0]:
from delta.tables import DeltaTable

silver_path = "abfss://silver@adlsnexpulse01.dfs.core.windows.net/inventory/"

if DeltaTable.isDeltaTable(spark, silver_path):
    dbutils.fs.rm(silver_path, True)
    print("Old Silver Inventory table removed.")
else:
    print("No existing Silver Inventory table found.")

Old Silver Inventory table removed.


In [0]:
(
    final_valid_df.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .save(silver_path)
)

print("Initial Silver Inventory table created.")

Initial Silver Inventory table created.


In [0]:
silver_inventory_table = DeltaTable.forPath(
    spark,
    silver_path
)

(
    silver_inventory_table.alias("t")
    .merge(
        final_valid_df.alias("s"),
        "t.product_id = s.product_id AND "
        "t.warehouse_id = s.warehouse_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Inventory Silver MERGE completed successfully.")

Inventory Silver MERGE completed successfully.


In [0]:
(
    spark.read
    .format("delta")
    .load(silver_path)
    .groupBy("product_id", "warehouse_id")
    .count()
    .filter("count > 1")
    .show()
)

+----------+------------+-----+
|product_id|warehouse_id|count|
+----------+------------+-----+
+----------+------------+-----+



In [0]:
(
    quarantine_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .save(quarantine_path)
)

print("Inventory quarantine written successfully.")

Inventory quarantine written successfully.


In [0]:
reconciled_total = (
    final_valid_count
    + quarantined_count
    + duplicates_removed_count
)

print("========== INVENTORY SILVER RECONCILIATION ==========")
print(f"Bronze rows:             {bronze_count}")
print(f"Valid before dedup:      {pre_dedup_valid_count}")
print(f"State rows removed:      {duplicates_removed_count}")
print(f"Final inventory states:  {final_valid_count}")
print(f"Quarantined rows:        {quarantined_count}")
print(f"Reconciled total:        {reconciled_total}")

assert bronze_count == reconciled_total

print()
print("PASSED: Inventory reconciliation")

========== INVENTORY SILVER RECONCILIATION ==========
Bronze rows:             60
Valid before dedup:      59
State rows removed:      4
Final inventory states:  55
Quarantined rows:        1
Reconciled total:        60

PASSED: Inventory reconciliation


In [0]:
silver_inventory_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)

invalid_silver_count = (
    silver_inventory_df
    .filter(col("failure_reason").isNotNull())
    .count()
)

print(f"Invalid rows in Silver Inventory: {invalid_silver_count}")

assert invalid_silver_count == 0

print("PASSED: Silver Inventory quality check")

Invalid rows in Silver Inventory: 0
PASSED: Silver Inventory quality check


In [0]:
display(
    quarantine_df.select(
        "event_id",
        "warehouse_id",
        "stock_quantity",
        "failure_reason"
    ).limit(20)
)

event_id,warehouse_id,stock_quantity,failure_reason
evt_45a6b429c7,WH_04,-60,invalid_stock_quantity


In [0]:
print("====================================================")
print("NEXPULSE — INVENTORY SILVER STEP 6")
print("====================================================")

print(f"Bronze Inventory rows:     {bronze_count}")
print(f"Valid before dedup:        {pre_dedup_valid_count}")
print(f"State rows removed:        {duplicates_removed_count}")
print(f"Final inventory states:    {final_valid_count}")
print(f"Quarantined rows:          {quarantined_count}")
print(f"Reconciled total:          {reconciled_total}")

print("----------------------------------------------------")
print("Inventory MERGE key:")
print("product_id + warehouse_id = one current state")

print("----------------------------------------------------")
print("MERGE behavior:")
print("Existing product/warehouse → UPDATE")
print("New product/warehouse      → INSERT")

print("----------------------------------------------------")
print("STATUS: INVENTORY SILVER STEP 6 COMPLETE")

NEXPULSE — INVENTORY SILVER STEP 6
Bronze Inventory rows:     60
Valid before dedup:        59
State rows removed:        4
Final inventory states:    55
Quarantined rows:          1
Reconciled total:          60
----------------------------------------------------
Inventory MERGE key:
product_id + warehouse_id = one current state
----------------------------------------------------
MERGE behavior:
Existing product/warehouse → UPDATE
New product/warehouse      → INSERT
----------------------------------------------------
STATUS: INVENTORY SILVER STEP 6 COMPLETE


In [0]:
try:
    log_pipeline_run(
        pipeline_name="silver_inventory",
        source="inventory",
        start_time=run_start,
        records_read=bronze_count,
        records_written=final_valid_count,
        records_quarantined=quarantined_count,
        records_deduplicated=duplicates_removed_count,
        status="success"
    )
except Exception as e:
    log_pipeline_run(
        pipeline_name="silver_inventory",
        source="inventory",
        start_time=run_start,
        status="failed",
        error_message=str(e)[:500]
    )
    raise

[audit] logged run 6694f9b4-7a1c-45c5-b02c-ed97a65f3858 (success), latency=118.3s


In [0]:
# ============================================================
# STEP 9 — VERIFY ALL SILVER PIPELINE AUDIT RUNS
# ============================================================

from pyspark.sql.functions import col

audit_df = (
    spark.read
    .format("delta")
    .load(audit_table_path)
)

display(
    audit_df
    .orderBy(col("start_time").desc())
    .limit(10)
)

pipeline_run_id,pipeline_name,source,start_time,end_time,records_read,records_written,records_quarantined,records_deduplicated,status,error_message,processing_latency_seconds
6694f9b4-7a1c-45c5-b02c-ed97a65f3858,silver_inventory,inventory,2026-08-22T14:07:47.299018Z,2026-08-22T14:09:45.578714Z,60,55,1,4,success,null,118.279696
858da511-6e80-45b0-8e19-6dffbc837814,audit_helper_test,test,2026-08-22T14:07:31.145758Z,2026-08-22T14:07:31.145801Z,1,1,0,0,success,null,4.3E-5
03ad41e4-c51f-4484-9434-085d172838bd,silver_payments,payments,2026-08-22T14:04:19.793698Z,2026-08-22T14:05:22.235177Z,0,0,0,0,failed,name 'bronze_count' is not defined,62.441479
0d629ba8-10b0-4a34-b61a-84ffaf948112,silver_payments,payments,2026-08-22T14:04:19.793698Z,2026-08-22T14:06:05.296975Z,60,59,0,1,success,null,105.503277
cb9c6dd1-e926-422b-98d4-ea4278ca7505,audit_helper_test,test,2026-08-22T14:04:01.275699Z,2026-08-22T14:04:01.275745Z,1,1,0,0,success,null,4.6E-5
48d2f909-4217-4bad-ad68-d875654dfcdf,silver_orders,orders,2026-08-22T13:57:41.842647Z,2026-08-22T13:58:07.137894Z,60,52,7,1,success,null,25.295247
c46bcb24-e431-4f98-a94a-1eb65418f70e,audit_helper_test,test,2026-08-22T13:57:22.498773Z,2026-08-22T13:57:22.498845Z,1,1,0,0,success,null,7.2E-5
cb24b51a-43ca-48aa-8379-89d3340c939f,audit_helper_test,test,2026-08-22T13:56:53.804145Z,2026-08-22T13:56:53.804197Z,1,1,0,0,success,null,5.2E-5
05a76169-5b03-452f-8fcc-a4542306f6b6,audit_helper_test,test,2026-08-22T13:49:42.089198Z,2026-08-22T13:49:42.089323Z,1,1,0,0,success,null,1.25E-4
c60d0f6b-bfd7-4d9e-aa3c-d735c17d74ca,audit_helper_test,test,2026-08-22T13:41:27.598137Z,2026-08-22T13:41:27.598214Z,1,1,0,0,success,null,7.7E-5
